In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# GCN + PRIMEVUL (CLASS-IMBALANCE FIXED)
# Dataset: primevul-gcn0
# ============================================================

import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH
# ============================================================
DATASET_PATH = "/kaggle/input/primevul-gcn0"
print("\nFiles in dataset:")
print(os.listdir(DATASET_PATH))

# ============================================================
# LOAD PRIMEVUL JSONL
# ============================================================
def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            data.append({
                "code": obj["func"],
                "label": int(obj["target"])
            })
    return pd.DataFrame(data)

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("\nLabel distribution (train):")
print(train_df["label"].value_counts())

# ============================================================
# CLASS WEIGHTS (🔥 MOST IMPORTANT FIX)
# ============================================================
counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [counts[1] / counts.sum(), counts[0] / counts.sum()],
    dtype=torch.float
).to(device)

print("\nClass weights:", class_weights)

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
MAX_LEN = 200
WINDOW = 4   # co-occurrence window

# ============================================================
# GRAPH DATASET
# ============================================================
class PrimeVulGraphDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def build_adj(self, L):
        adj = torch.zeros((L, L))
        for i in range(L):
            for j in range(max(0, i-WINDOW), min(L, i+WINDOW+1)):
                adj[i, j] = 1
        return adj

    def __getitem__(self, idx):
        tokens = tokenizer.encode(
            self.codes[idx],
            truncation=True,
            max_length=MAX_LEN,
            add_special_tokens=False
        )

        L = len(tokens)
        x = torch.tensor(tokens, dtype=torch.long)
        adj = self.build_adj(L)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, adj, label

    def __len__(self):
        return len(self.codes)

def collate_fn(batch):
    xs, adjs, labels = zip(*batch)
    max_len = max(len(x) for x in xs)

    X = torch.zeros(len(xs), max_len, dtype=torch.long)
    A = torch.zeros(len(xs), max_len, max_len)

    for i in range(len(xs)):
        X[i, :len(xs[i])] = xs[i]
        A[i, :adjs[i].shape[0], :adjs[i].shape[1]] = adjs[i]

    return X.to(device), A.to(device), torch.tensor(labels).to(device)

train_loader = DataLoader(
    PrimeVulGraphDataset(train_df),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    PrimeVulGraphDataset(test_df),
    batch_size=16,
    collate_fn=collate_fn
)

# ============================================================
# GCN MODEL
# ============================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        deg = adj.sum(dim=-1, keepdim=True) + 1e-6
        x = torch.bmm(adj, x) / deg
        return self.fc(x)

class GCN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gcn1 = GCNLayer(embed_dim, 128)
        self.gcn2 = GCNLayer(128, 128)
        self.fc = nn.Linear(128, 2)

    def forward(self, x, adj):
        x = self.embed(x)
        x = F.relu(self.gcn1(x, adj))
        x = F.relu(self.gcn2(x, adj))
        x = x.mean(dim=1)   # graph-level pooling
        return self.fc(x)

model = GCN(tokenizer.vocab_size).to(device)

# ============================================================
# TRAINING SETUP
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
EPOCHS = 8

# ============================================================
# TRAIN
# ============================================================
print("\nTraining GCN on PrimeVul...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X, A, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X, A), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ============================================================
# EVALUATION
# ============================================================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X, A, y in test_loader:
        preds = torch.argmax(model(X, A), dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nPrediction distribution:", np.unique(y_pred, return_counts=True))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n===== GCN PRIMEVUL RESULTS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))
print("FPR      :", fp / (fp + tn) if (fp + tn) > 0 else 0)
print("Confusion Matrix:", tn, fp, fn, tp)


Device: cuda
GPU: Tesla T4

Files in dataset:
['primevul_test_paired.jsonl', 'primevul_train_paired.jsonl', 'primevul_valid_paired.jsonl']

Label distribution (train):
label
1    3789
0    3789
Name: count, dtype: int64

Class weights: tensor([0.5000, 0.5000], device='cuda:0')


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


Training GCN on PrimeVul...
Epoch 1/8 | Loss: 0.6946
Epoch 2/8 | Loss: 0.6935
Epoch 3/8 | Loss: 0.6935
Epoch 4/8 | Loss: 0.6933
Epoch 5/8 | Loss: 0.6932
Epoch 6/8 | Loss: 0.6932
Epoch 7/8 | Loss: 0.6932
Epoch 8/8 | Loss: 0.6930

Prediction distribution: (array([0, 1]), array([746, 124]))

===== GCN PRIMEVUL RESULTS =====
Accuracy : 0.506896551724138
Precision: 0.5241935483870968
Recall   : 0.14942528735632185
F1 Score : 0.23255813953488372
FPR      : 0.135632183908046
Confusion Matrix: 376 59 370 65
